# Compare SED outputs from different runs

In [ ]:
import os
import glob
import numpy as np
import pickle as pkl
import pandas as pd
import prospect.io.read_results as reader
from prospect.utils.plotting import get_percentiles, get_best
from corner import quantile
import h5py


from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import WMAP9 as cosmo
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.sources import FastStepBasis
from astropy.cosmology import Planck18 as cosmo
from prospect.models.sedmodel import PolySpecModel, SpecModel


# Prospector Loading Function

In [ ]:
def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    # Load the h5 file for the given galaxy ID
    h5_files = glob.glob(os.path.join(prosp_dir, f'*{galaxy_id}*.h5'))
    try:
        h5_file = h5_files[0]
        print(f"Loading Prospector output file: {h5_file}")
    except IndexError:
        print(f"No PROSPECTOR results found for objid {galaxy_id}.")
        return None

    # Load PROSPECTOR results
    results, obs, model = reader.results_from(h5_file)
        
    # Now we have to exclude the last 3 parameters from the fit
    map_parameters = get_best(results)
    
    # Extract labels for parameters that were "free" (fitted)
    labels = map_parameters[0]

    # Build the MAP dictionary
    MAP = {}
    for a,b in zip(map_parameters[0], map_parameters[1]):
        MAP[a] = b
    
    # Extract chains, weights and the MAP index
    chain = results['chain']
    weights = results['weights']
    imax = np.argmax(results['lnprobability'])
    
    data = {
            'meta': {'labels': map_parameters[0], 'map_idx': imax, 'weights': weights},
            'params': {}
        }

    perc = get_percentiles(results, [16, 50, 84])    
    
    for i, name in enumerate(map_parameters[0]):
            # Use the flattened chain for statistics
            param_samples = chain[:, i]
            
            if name == 'dust2':         # convert optical depth to mag
                data['params'][name] = {
                'samples': param_samples * 1.086,
                'map': MAP[name] * 1.086,   # Use the value from the best vector directly
                'q16': perc[name][0] * 1.086,
                'q50': perc[name][1] * 1.086,
                'q84': perc[name][2] * 1.086
            }
                
            else:
                data['params'][name] = {
                    'samples': param_samples,
                    'map': MAP[name],   # Use the value from the best vector directly
                    'q16': perc[name][0],
                    'q50': perc[name][1],
                    'q84': perc[name][2]
                }
    return data

phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

data = load_prospector_results(12717, prosp_dir)

# Bagpipes Loading Function

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

example_id = 7102

def load_bagpipes_results(galaxy_id, bagp_dir):
    """Function to load Bagpipes result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        bagp_dir (str): The directory containing the Bagpipes output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = os.path.join(bagp_dir, f'{galaxy_id}.h5')    # Only one file per galaxy
    
    print(f"Loading Bagpipes output file: {file}")
    
    with h5py.File(file, 'r') as results:
        
        # Get redshift of the source
        fit_str = results.attrs['fit_instructions']
        fit = eval(fit_str, {"np": np, "array": np.array})
        zred = fit['redshift']
        
        # Extract the sampling results
        chain = results['samples2d']
        
        # Get maximum likelihood inde
        imax = np.argmax(results['lnlike'])
        
        # Manually extracted the labels from the results
        labels = ['dsfr1', 'dsfr2', 'dsfr3', 'dsfr4', 'dsfr5', 'dsfr6', 'logmass', 'logzsol', 'dust2', 
                    'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
        
        # Build the data structure
        data = {
            'meta': {'labels': labels, 'map_idx': imax, 'weights': None},
            'params': {}
        }
        
        # Loop through each parameter
        for i, name in enumerate(labels):
            samples = chain[:, i]
            q16, q50, q84 = quantile(samples, [0.16, 0.5, 0.84], weights=None)
            
            if name == 'logzsol':   # Convert metallicity to log
                data['params'][name] = {
                    'samples': np.log10(samples),
                    'map': np.log10(samples[imax]),
                    'q16': np.log10(q16),
                    'q50': np.log10(q50),
                    'q84': np.log10(q84)
                }
            
            else:    
                data['params'][name] = {
                    'samples': samples,
                    'map': samples[imax],
                    'q16': q16,
                    'q50': q50,
                    'q84': q84
                }

        data['params']['zred'] = {'samples': None, 'map': zred, 'q16': zred, 'q50': zred, 'q84': zred}
        
    return data
    
example_res = load_bagpipes_results(21424, bagp_dir)
print(example_res)


# Load results and store them in pickle files

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/fesc_with_miri/pickles'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/fesc_with_miri/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        p_data = load_prospector_results(gal_id, prosp_dir)
        b_data = load_bagpipes_results(gal_id, bagp_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'prospector': p_data,
        'bagpipes': b_data
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

# Check double-peaks

In [ ]:
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

pickle_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

def find_modes(samples, weights=None, prominence=0.05, mass_threshold=0.2):
    """
    Returns peaks that contain at least 'mass_threshold' (e.g., 20%) of posterior mass.
    """
    if weights is None: weights = np.ones(len(samples))
    
    # 1. Find peaks using KDE
    kde = gaussian_kde(samples, weights=weights)
    x_grid = np.linspace(samples.min(), samples.max(), 500)
    density = kde.evaluate(x_grid)
    peak_indices, _ = find_peaks(density, prominence=prominence * density.max())
    peak_values = x_grid[peak_indices]
    peak_densities = density[peak_indices]
    
    if len(peak_values) == 0: return []
    
    # 2. Assign each sample to its closest peak
    # (Reshaping for broadcasting: samples [N,1] - peaks [1, P])
    dist = np.abs(samples[:, np.newaxis] - peak_values[np.newaxis, :])
    closest_peak_idx = np.argmin(dist, axis=1)
    
    # 3. Calculate mass fraction for each peak
    total_weight = np.sum(weights)
    significant_data = []
    
    for i in range(len(peak_values)):
        peak_mass = np.sum(weights[closest_peak_idx == i]) / total_weight
        
        if peak_mass >= mass_threshold:
            # We store the value AND the density so we can sort later
            significant_data.append((peak_values[i], peak_densities[i]))
            
    if not significant_data: return np.array([])
    
    # 4. Sort by density (index 1) in descending order
    significant_data.sort(key=lambda x: x[1], reverse=True)
    
    # Return just the values, now guaranteed to be density-sorted
    return np.array([item[0] for item in significant_data])

def check_for_overlap(peaks1, peaks2, tolerance=0.1):
    """
    Checks if any peak in code 1 is 'near' a peak in code 2.
    tolerance: how close peaks need to be (in normalized parameter space)
    """
    overlaps = []
    for p1 in peaks1:
        for p2 in peaks2:
            if abs(p1 - p2) < tolerance:
                overlaps.append((p1, p2))
    return overlaps

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'b': [], 'p': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists

tolerances = {
    'logmass': 0.2,       # 0.2 dex is reasonable
    'logzsol': 0.3,       # 0.3 dex allows for template differences
    'dust2': 0.5,         # Av is often degenerate, so give it more slack
    'duste_gamma': 0.1,
    'dust_index': 0.2,
    'duste_qpah': 1.5,    # Notice this is large because your plot range is 0-10
    'duste_umin': 4.0,    # Large range, large tolerance
    'gas_logu': 0.4
}

file = files[0]

results = {}

file = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/pickles/21218_comp.pkl"
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        objid = data["id"]
        bagp = data['bagpipes']['params']
        prosp = data['prospector']['params']
        
        b_weights = data['bagpipes']['meta']['weights']
        p_weights = data['prospector']['meta']['weights']
        
        results[objid] = {}
        
        for label, tolerance in tolerances.items():
            bag_peaks = find_modes(bagp[label]["samples"], weights=b_weights)
            pros_peaks = find_modes(prosp[label]["samples"], weights=p_weights)
            
            bag_best = bagp[label]["q50"]
            pros_best = prosp[label]["map"]
            
            if len(bag_peaks) == 0:
                gap_bag = 0.0
            else:
                gap_bag = np.abs(bag_peaks[0] - bag_best)    
            if len(pros_peaks) == 0:
                pros_bag = 0.0
            else:   
                gap_pros = np.abs(pros_peaks[0] - pros_best)
            
            if gap_bag > tolerance:
                #print(f"Parameter {label}: Bagpipes median is lies {gap_bag} from largest peak.")
                bag_offset = True
            else:
                bag_offset = False
            if gap_pros > tolerance:
                #print(f"Parameter {label}: Prospector MAP is lies {gap_pros} from largest peak.")
                pros_offset = True
            else:
                pros_offset = False
            
            # DIVIDE GALAXIES INTO CATEGORIES
            overlaps = check_for_overlap(bag_peaks, pros_peaks, tolerance=tolerance)
            #print(f"{label}: {overlaps}")
                
            # Correct QC Flag Assignment
            if (len(overlaps) > 0) and not (bag_offset or pros_offset):
                qc_flag = 1
            elif (len(overlaps) == 0) and not (bag_offset or pros_offset):
                qc_flag = 2
            elif (len(overlaps) > 0) and (bag_offset or pros_offset):
                qc_flag = 3
            else:
                qc_flag = 4
                    
            # Store into the sub-dictionary
            results[objid][label] = {
                "qc_flag": qc_flag, 
                "bag_offset": gap_bag,
                "pros_offset": gap_pros,
                "has_overlap": len(overlaps) > 0
            }
        
import pandas as pd
df = pd.DataFrame.from_dict({(i, j): results[i][j] 
                           for i in results.keys() 
                           for j in results[i].keys()}, orient='index')
        
print(df)
        
        

Save the dataframe for future analysis

In [ ]:
df_new = df.reset_index()
#df_new.rename(columns={'index': 'id'}, inplace=True)
df_new.rename(columns={"level_0": "id", "level_1": "param", "category": "qc_flag"}, inplace=True)

df_new.to_csv('/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.csv', index=False)

Analyse the dataframe now

In [ ]:
import pandas as pd

# 1. Load the data
df3 = pd.read_csv('/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.csv', index_col=0)

# Convert back to dictionary
results = {}
for objid, row in df3.iterrows():
    if objid not in results:
        results[objid] = {}
    
    label = row['param']
    qc_flag = row['qc_flag']
    gap_bag = row["bag_offset"]
    gap_pros = row["pros_offset"]
    has_overlaps = row["has_overlap"]
    
    results[objid][label] = {
        "qc_flag": qc_flag, 
        "bag_offset": gap_bag,
        "pros_offset": gap_pros,
        "has_overlap": len(overlaps) > 0
    }

# Got it, now this rebuilt my dictionary like I saved it!
print(results)

Plotting the `qc_flag` param

In [ ]:
from matplotlib.figure import SubFigure
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert the dict to a DataFrame
# This flattens the dictionary: results[objid][label]['qc_flag']
#data_for_heatmap = {
#    objid: {label: values['qc_flag'] for label, values in labels.items()}
#    for objid, labels in results.items()
#}

data_for_heatmap = {}
for objid, labels in results.items():
    data_for_heatmap[objid] = {}
    for label, values in labels.items():
        data_for_heatmap[objid][label] = values['qc_flag']

df_heatmap = pd.DataFrame.from_dict(data_for_heatmap, orient='index')

# Sort rows by index (Galaxy ID) ascending
df_heatmap = df_heatmap.sort_index()

# 2. Setup the Heatmap
fig, ax = plt.subplots(figsize=(5, 20))

# Use a discrete color map (1=Green, 2=Yellow, 3=Orange, 4=Red)
cmap = sns.color_palette(["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], as_cmap=True)

sns.heatmap(df_heatmap, 
            cbar=False,
            cmap=cmap, 
            annot=True, 
            #cbar_kws={'ticks': [1, 2, 3, 4]}, 
            vmin=0.5, vmax=4.5)

# Calculate stats for the labels
means = df_heatmap.mean()
medians = df_heatmap.median()

# Create a copy to add summary rows
df_plot = df_heatmap.copy()
df_plot.loc['MEAN'] = means
df_plot.loc['MEDIAN'] = medians

# Create labels with stats (e.g., "logmass\nμ=1.2, md=1.0")
#labels = [f"{col}\nμ={m:.2f}\nmed={med:.1f}" for col, m, med in zip(df_heatmap.columns, means, medians)]

# Move the x-axis ticks to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

ax.set_xticks(np.arange(df_heatmap.shape[1])+0.5)
ax.set_xticklabels(labels, rotation=45, ha='left', fontsize=12)

legend = "Quality flagging criteria:\n"
legend += "1: Peaks agree AND no offset (Peak - MAP)\n"
legend += "2: Peaks don't agree AND no offset (Peak - MAP)\n"
legend += "3: Peaks agree AND significant offset (Peak - MAP)\n"
legend += "4: Peaks don't agree AND significant offset (Peak - MAP)\n\n"

labels = ""
for col, m, med in zip(df_heatmap.columns, means, medians):
    labels += f"{col}: μ={m:.2f}, med={med:.1f}\n"

stats_legend = legend + labels

plt.figtext(0.95, 0.8, stats_legend, verticalalignment='center')
#plt.subplots_adjust(right=0.8) # Make room for text on the right
#ax.text(x=1.0, y=0.5, s=stats_legend)
# Clean up axes
#ax.set_xlabel("")

plt.title("Posterior Agreement - Bagpipes vs. Prospector", fontsize=14)
#plt.xlabel("Parameter")
#plt.ylabel("Galaxy ID")
plt.tight_layout()
fig_path = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

# Plot parameter comparison Bagpipes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats
import glob
import os

pickle_dir = './comparison/no_fesc_with_miri/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'b': [], 'p': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        bagp = data['bagpipes']['params']
        prosp = data['prospector']['params']
        
        for p_key in collected_data.keys():
            if p_key in bagp and p_key in prosp:
                # Store the MAP value for this galaxy
                collected_data[p_key]['b'].append(bagp[p_key]['q50'])
                collected_data[p_key]['p'].append(prosp[p_key]['map'])

fig_path = './comparison/no_fesc_with_miri/withorwithout_best.png'

# Define the parameters we want to plot
# Format: (conceptual_name, df_prefix, display_label)
params = [
    ('logmass', r'$\log_{10}(M_*/M_\odot)$'),
    ('logzsol', r'$\log_{10}(Z/Z_\odot)$'),
    ('dust2', r'$A_V$ [mag]'),
    ('gas_logu', r'$\log_{10}(U)$'),
    ('duste_qpah', r'$q_{PAH}$ [%]'),
    ('duste_umin', r'Dust $U_{min}$'),
    ('duste_gamma', r'Dust $\gamma$'),
    ('dust_index', r'$n_{dust}$')
]

n_params = len(params)
fig, axes = plt.subplots(2, n_params//2, figsize=(14, 8))

axes = axes.flatten()

for i, (col, label) in enumerate(params):
    ax = axes[i]
    
    x = np.array(collected_data[col]['b'])
    y = np.array(collected_data[col]['p'])
    
    if len(x) == 0: continue

    # Calculate limits
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    pad = (vmax - vmin) * 0.1
    vmin, vmax = vmin - pad, vmax + pad

    # Plot 1:1 Line
    ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)
    
    # Labels and formatting
    ax.set_title(f'{label}', fontsize=14)
    ax.set_xlabel(f'Bagpipes', fontsize=12)
    ax.set_ylabel(f'Prospector', fontsize=12)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Calculate and display Mean Offset
    if i in [0, 2]:
        offset = np.nanmedian(y - x)
        scatter = median_abs_deviation(y - x)
        ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    else:
        corr_coef, p_value = stats.pearsonr(x, y)
        ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


# Plot Dual Corner

In [ ]:
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl
    
# Increase tick label size and thickness
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['axes.linewidth'] = 1.5  # Makes the box frames thicker

def plot_dual_corner(galaxy_id, comparison_dir):
    
    file = os.path.join(comparison_dir, f'pickles/{galaxy_id}_comp.pkl')  
    with open(file, 'rb') as f:
        data = pkl.load(f)
        
        # Load the individual fit results
        b_data = data['bagpipes']
        p_data = data['prospector']
    
    # Define internal keys and display labels
    # Make sure these match the keys in your b_data['params'] and p_data['params']
    plot_keys = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    labels = [r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", 
              r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", 
              r"Dust $U_{min}$", r"$\log_{10}(U)$"]

    # Extract and Transform Samples
    # Bagpipes
    b_samps = np.array([b_data['params'][k]['samples'] for k in plot_keys]).T
    # Prospector (Applying the Av conversion factor 1.086 to the 3rd column: index 2)
    p_samps = np.array([p_data['params'][k]['samples'] for k in plot_keys]).T
    
    samples_list = [b_samps, p_samps]
    sample_labels = ["Bagpipes", "Prospector"]
    colors = ["orange", "dodgerblue"]

    # Calculate Global Range (So both fits are visible)
    ndim = b_samps.shape[1]
    plot_range = []
    for dim in range(ndim):
        dim_min = min(np.nanmin(b_samps[:, dim]), np.nanmin(p_samps[:, dim]))
        dim_max = max(np.nanmax(b_samps[:, dim]), np.nanmax(p_samps[:, dim]))
        
        plot_range.append([dim_min, dim_max])

    # 4. Handle Weights 
    # Prospector weights from Dynesty
    p_weights = p_data['meta']['weights']
    # Bagpipes weights (Usually None/Equal, so create ones)
    b_weights = np.ones(len(b_samps))
    
    # Normalize weights so histograms have comparable heights
    # (Matching the logic from your provided demo script)
    b_weights *= (len(p_samps) / len(b_samps))

    # 5. Base Corner Settings
    shared_kwargs = dict(
        labels=labels,
        range=plot_range,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False, # We disable automatic titles to avoid overlaps
        max_n_ticks=3,
        hist_kwargs=dict(density=True)
    )

    # 6. Plotting
    # First: Bagpipes
    fig = corner.corner(
        b_samps,
        labels=labels,
        range=plot_range,
        color="orange",
        label_kwargs={"fontsize": 14},
        weights=b_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'orange', 'linewidth': 2, 'density': True}
    )

    # Second: Prospector
    # We turn off 'fill_contours' for the second one so we can see through it
    corner.corner(
        p_samps,
        fig=fig,
        range=plot_range,
        color="dodgerblue",
        label_kwargs={"fontsize": 14}, 
        weights=p_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True, 
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'dodgerblue', 'linewidth': 2, 'density': True}
    )

    # 7. Add Legend and Title
    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i], lw=4)
            for i in range(len(colors))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    
    ndim = b_samps.shape[1]
    axes = np.array(fig.axes).reshape((ndim, ndim))
    
    for i in range(ndim):
        ax = axes[i, i]
        
        # 2. Extract stats for Bagpipes (Orange)
        b_p = b_data['params'][plot_keys[i]]
        b_val, b_plus, b_minus = b_p['q50'], b_p['q84'] - b_p['q50'], b_p['q50'] - b_p['q16']
        
        # 3. Extract stats for Prospector (Black/Blue)
        p_p = p_data['params'][plot_keys[i]]
        p_val, p_plus, p_minus = p_p['q50'], p_p['q84'] - p_p['q50'], p_p['q50'] - p_p['q16']
        
        # Special Case: If it's Av (index 2), apply the 1.086 scale to the text labels too
        if i == 2:
            b_val, b_plus, b_minus = b_val, b_plus, b_minus # Bagpipes is already Av
            p_val, p_plus, p_minus = p_val*1.086, p_plus*1.086, p_minus*1.086
        
        # 4. Create the strings
        # Use \text{} or raw strings to handle the LaTeX formatting
        b_str = f"${b_val:.2f}^{{+{b_plus:.2f}}}_{{-{b_minus:.2f}}}$"
        p_str = f"${p_val:.2f}^{{+{p_plus:.2f}}}_{{-{p_minus:.2f}}}$"

        # 5. Set the title
        # We use a newline \n to stack them. Note: 'y' controls the vertical height.
        # 4. Place individual text objects (Manually colored)
        # x=0.5 centers it. y=1.02 and 1.15 stack them above the plot.
        ax.text(0.5, 1.15, b_str, color="orange", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        ax.text(0.5, 1.02, p_str, color="dodgerblue", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        # 6. Coloring the text
        # To get specific colors for specific lines of the title, 
        # we can use ax.annotate or just rely on the labels in the legend.
        # But for absolute clarity, we can color the whole title block:
        #ax.title.set_color('black') # Or 'darkgrey' to be neutral
    
    fig.suptitle(f"Bagpipes vs. Prospector\nGalaxy {galaxy_id}", fontsize=24, y=1.0)
    
    fig_path = os.path.join(comparison_dir, 'corner_plots', f'{galaxy_id}_corner.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()

# Create corner plots for the whole sample!

In [ ]:
comp_dir = './comparison/fesc_with_miri/'

pickle_files = glob.glob(os.path.join(comp_dir, 'pickles', '*_comp.pkl'))

for p in pickle_files:
    objid = int(os.path.basename(p).split('_comp.pkl')[0])  # Extract the galaxy ID from the filename
    plot_dual_corner(objid, comp_dir)
    print(f"Plotted corner plot for galaxy ID: {objid}")

# Look at a single Bagpipes output

In [ ]:
def plot_bagpipes_corner(galaxy_id, b_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([b_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=b_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="Orange",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    
    fig.suptitle(f"Galaxy {galaxy_id} Bagpipes", fontsize=16)
    plt.show()

example_res = load_bagpipes_results(21424, bagp_dir)

plot_bagpipes_corner(21424, example_res)

# Plot Prospector Corner Plot

In [ ]:
def plot_prospector_corner(galaxy_id, p_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([p_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=p_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="black",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    fig.suptitle(f"Galaxy {galaxy_id} Prospector", fontsize=16)
    plt.show()

example_res = load_prospector_results(21424, prosp_dir)

plot_prospector_corner(21424, example_res)

# Compare Prospector with and without MIRI

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/prospector/pickles'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

miri_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
no_miri_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.2/sourcephotonly_v2.0.2/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        data1 = load_prospector_results(gal_id, miri_dir)
        data2 = load_prospector_results(gal_id, no_miri_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'miri': data1,
        'no_miri': data2
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

Plot the parameters before and after

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats

pickle_dir = './comparison/prospector/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'with': [], 'without': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        miri = data['miri']['params']
        no_miri = data['no_miri']['params']
        
        for p_key in collected_data.keys():
            if p_key in miri and p_key in no_miri:
                # Store the MAP value for this galaxy
                collected_data[p_key]['with'].append(miri[p_key]['q50'])
                collected_data[p_key]['without'].append(no_miri[p_key]['q50'])

fig_path = './comparison/prospector/plots/withorwithout_median.png'

# Define the parameters we want to plot
# Format: (conceptual_name, df_prefix, display_label)
params = [
    ('logmass', r'$\log_{10}(M_*/M_\odot)$'),
    ('logzsol', r'$\log_{10}(Z/Z_\odot)$'),
    ('dust2', r'$A_V$ [mag]'),
    ('gas_logu', r'$\log_{10}(U)$'),
    ('duste_qpah', r'$q_{PAH}$ [%]'),
    ('duste_umin', r'Dust $U_{min}$'),
    ('duste_gamma', r'Dust $\gamma$'),
    ('dust_index', r'$n_{dust}$')
]

n_params = len(params)
fig, axes = plt.subplots(2, n_params//2, figsize=(14, 8))

axes = axes.flatten()

for i, (col, label) in enumerate(params):
    ax = axes[i]
    
    x = np.array(collected_data[col]['with'])
    y = np.array(collected_data[col]['without'])
    
    if len(x) == 0: continue

    # Calculate limits
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    pad = (vmax - vmin) * 0.1
    vmin, vmax = vmin - pad, vmax + pad

    # Plot 1:1 Line
    ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)
    
    # Labels and formatting
    ax.set_title(f'{label}', fontsize=14)
    ax.set_xlabel(f'With MIRI', fontsize=12)
    ax.set_ylabel(f'No MIRI', fontsize=12)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Calculate and display Mean Offset
    if i in [0, 2]:
        offset = np.nanmedian(y - x)
        scatter = median_abs_deviation(y - x)
        ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    else:
        corr_coef, p_value = stats.pearsonr(x, y)
        ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


# Dust2 with and without MIRI

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats
import glob
import os

pickle_dir = './comparison/prospector/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

dust2 = {'id': [], 'with': [], 'without': []}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        dust2_miri = data['miri']['params']['dust2']['q50']*1.086
        dust2_no_miri = data['no_miri']['params']['dust2']['q50']*1.086
        
        dust2['id'].append(data['id'])
        dust2['with'].append(dust2_miri)
        dust2['without'].append(dust2_no_miri)
        
        if data['id'] == 7549:
            print(f"Galaxy {data['id']} - Dust2 with MIRI: {dust2_miri:.3f}, without MIRI: {dust2_no_miri:.3f}")

N = len(dust2['with'])
k = int(1 + np.log2(N))*2

range = [min(min(dust2['with']), min(dust2['without'])), max(max(dust2['with']), max(dust2['without']))]

fig, ax = plt.subplots(figsize=(5, 5))
ax.hist(dust2['with'], bins=k, alpha=0.5, label='With MIRI', color='dodgerblue', range=range)#, edgecolor='black')
ax.hist(dust2['without'], bins=k, alpha=0.5, label='Without MIRI', color='coral', range=range)#, edgecolor='black')
#ax.vlines(np.median(dust2['with']), ymin=0, ymax=ax.get_ylim()[1], color='dodgerblue', linestyle='--', label='Median With MIRI')
#ax.vlines(np.median(dust2['without']), ymin=0, ymax=ax.get_ylim()[1], color='coral', linestyle='--', label='Median Without MIRI')
ax.set_xlabel(r'$A_V$', fontsize=14)
ax.set_ylabel('Frequency', fontsize=14)
ax.tick_params(labelsize=14)
ax.legend()
plt.title(r'$A_V$ with and without MIRI', fontsize=16)
plt.show()

Now let's look at the difference

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

deltas = np.array(dust2['with']) - np.array(dust2['without'])   # Difference in dust2 after adding MIRI
imax = np.argmax(deltas)
print(imax)
print(deltas[imax])
print(list(dust2['id'])[imax])

ratio = np.array(dust2['with']) / np.array(dust2['without'])

print(np.median(ratio))

ax.scatter(dust2['without'], deltas, alpha=0.7, color='dodgerblue', edgecolor='black', s=50)
ax.hlines(0, xmin=0, xmax=ax.get_xlim()[1], color='gray', linestyle='--', alpha=0.7, linewidth=2.5)
ax.set_ylim(-2.0, 2.0)
ax.set_xlabel(r'$A_V$ without MIRI', fontsize=14)
ax.set_ylabel(r'$\Delta$ ', fontsize=14)
ax.tick_params(labelsize=14)
plt.title(r'$\Delta$ = $A_V$ with MIRI - $A_V$ without MIRI', fontsize=16)
plt.savefig('./comparison/prospector/plots/dust2_delta_median.png', dpi=300, bbox_inches='tight')
plt.show()

Corner Plot for ID 7549

In [ ]:
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pickle as pkl
import matplotlib as mpl

# Increase tick label size and thickness
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['axes.linewidth'] = 1.5  # Makes the box frames thicker


def plot_dual_corner(galaxy_id, b_data, p_data):
    # 1. Define internal keys and display labels
    # Make sure these match the keys in your b_data['params'] and p_data['params']
    plot_keys = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    labels = [r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", 
              r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", 
              r"Dust $U_{min}$", r"$\log_{10}(U)$"]

    # 2. Extract and Transform Samples
    # Bagpipes
    b_samps = np.array([b_data['params'][k]['samples'] for k in plot_keys]).T
    # Prospector (Applying the Av conversion factor 1.086 to the 3rd column: index 2)
    p_samps = np.array([p_data['params'][k]['samples'] for k in plot_keys]).T
    
    samples_list = [p_samps, b_samps]
    sample_labels = ["No MIRI", "With MIRI"]
    colors = ["orange", "dodgerblue"]

    # 3. Calculate Global Range (So both fits are visible)
    ndim = b_samps.shape[1]
    plot_range = []
    for dim in range(ndim):
        dim_min = min(np.nanmin(b_samps[:, dim]), np.nanmin(p_samps[:, dim]))
        dim_max = max(np.nanmax(b_samps[:, dim]), np.nanmax(p_samps[:, dim]))
        # Add 5% padding
        span = dim_max - dim_min
        plot_range.append([dim_min, dim_max])

    # 4. Handle Weights 
    # Prospector weights from Dynesty
    p_weights = p_data['meta']['weights']
    # Bagpipes weights (Usually None/Equal, so create ones)
    b_weights = b_data['meta']['weights']
    
    if b_weights is None:
        b_weights = np.ones(len(b_samps))
        
    # Normalise weights so histograms have comparable heights
    # (Matching the logic from your provided demo script)
    b_weights *= (len(p_samps) / len(b_samps))

    # 5. Base Corner Settings
    shared_kwargs = dict(
        labels=labels,
        range=plot_range,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False, # We disable automatic titles to avoid overlaps
        max_n_ticks=3,
        hist_kwargs=dict(density=True)
    )

    # 6. Plotting
    # First: Bagpipes
    fig = corner.corner(
        p_samps,
        labels=labels,
        range=plot_range,
        color="orange",
        weights=p_weights,
        label_kwargs={"fontsize": 14},
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'orange', 'linewidth': 2, 'density': True}
    )

    # Second: Prospector
    # We turn off 'fill_contours' for the second one so we can see through it
    corner.corner(
        b_samps,
        fig=fig,
        range=plot_range,
        color="dodgerblue",
        weights=b_weights,
        label_kwargs={"fontsize": 14},
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True, 
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'dodgerblue', 'linewidth': 2, 'linestyle': '-', 'density': True, 'alpha': 0.7}
    )

    # 7. Add Legend and Title
    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i], lw=2)
            for i in range(len(colors))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    
    ndim = b_samps.shape[1]
    axes = np.array(fig.axes).reshape((ndim, ndim))
    
    for i in range(ndim):
        ax = axes[i, i]
        
        # 2. Extract stats for Bagpipes (Orange)
        b_p = b_data['params'][plot_keys[i]]
        b_val, b_plus, b_minus = b_p['q50'], b_p['q84'] - b_p['q50'], b_p['q50'] - b_p['q16']
        
        # 3. Extract stats for Prospector (Black/Blue)
        p_p = p_data['params'][plot_keys[i]]
        p_val, p_plus, p_minus = p_p['q50'], p_p['q84'] - p_p['q50'], p_p['q50'] - p_p['q16']
        
        # Special Case: If it's Av (index 2), apply the 1.086 scale to the text labels too
        if i == 2:
            b_val, b_plus, b_minus = b_val, b_plus, b_minus # Bagpipes is already Av
            p_val, p_plus, p_minus = p_val*1.086, p_plus*1.086, p_minus*1.086
        
        # 4. Create the strings
        # Use \text{} or raw strings to handle the LaTeX formatting
        b_str = f"${b_val:.2f}^{{+{b_plus:.2f}}}_{{-{b_minus:.2f}}}$"
        p_str = f"${p_val:.2f}^{{+{p_plus:.2f}}}_{{-{p_minus:.2f}}}$"

        # 5. Set the title
        # We use a newline \n to stack them. Note: 'y' controls the vertical height.
        # 4. Place individual text objects (Manually colored)
        # x=0.5 centers it. y=1.02 and 1.15 stack them above the plot.
        ax.text(0.5, 1.15, b_str, color="orange", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        ax.text(0.5, 1.02, p_str, color="dodgerblue", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        # 6. Coloring the text
        # To get specific colors for specific lines of the title, 
        # we can use ax.annotate or just rely on the labels in the legend.
        # But for absolute clarity, we can color the whole title block:
        #ax.title.set_color('black') # Or 'darkgrey' to be neutral
    
    fig.suptitle(f"Galaxy {galaxy_id} Comparison", fontsize=24, y=1.02)
    
    return fig


pickle_file = './comparison/prospector/pickles/10339_comp.pkl'

# 2. Loop through files and fill the lists
with open(pickle_file, 'rb') as f:
    data = pkl.load(f)
    with_miri = data['miri']
    no_miri = data['no_miri']
    
    print(f"Galaxy {data['id']} - Dust2 with MIRI: {with_miri['params']['dust2']['map']*1.086:.3f}, without MIRI: {no_miri['params']['dust2']['map']*1.086:.3f}")
    
fig = plot_dual_corner(10339, with_miri, no_miri)

# Compare mock galaxy fits

New corner plot function

In [ ]:
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl
    
# Increase tick label size and thickness
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['axes.linewidth'] = 1.5  # Makes the box frames thicker

def plot_corner_mock(galaxy_id, data1, label1, data2, label2, mock_data, save_dir, scale_dust1=False, scale_dust2=True):
    """Figure to plot corner plot of two posterior distributions, given that they are stored in the same format.

    Args:
        galaxy_id (int): ID of the galaxy
        data1 (dict): Output data of the first fit
        label1 (str): String describing the first fit
        data2 (dict): Output data of the second fit
        label2 (str): String describing the second fit
        mock_data (dict): Dictionary containing the real galaxy properties
        save_dir (str): Output directory for the figure
    
    Note:
        Data 1 is displayed in Orange
        Data 2 is displayed in Blue
    """
    # Define internal keys and display labels
    plot_keys = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    labels = [r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", 
              r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", 
              r"Dust $U_{min}$", r"$\log_{10}(U)$"]

    # Extract and Transform Samples
    a_samps = np.array([data1['params'][k]['samples'] for k in plot_keys]).T
    b_samps = np.array([data2['params'][k]['samples'] for k in plot_keys]).T
    
    a_weights = data1['meta'].get('weights') if 'meta' in data1 else None
    b_weights = data2['meta'].get('weights') if 'meta' in data2 else None
    
    a_weights = np.ones(len(a_samps)) if a_weights is None else a_weights
    b_weights = np.ones(len(b_samps)) if b_weights is None else b_weights
    
    samples_list = [a_samps, b_samps]
    sample_labels = [label1, label2]
    colors = ["orange", "dodgerblue"]

    # Calculate Global Range (So both fits are visible)
    ndim = b_samps.shape[1]
    plot_range = []
    for dim in range(ndim):
        dim_min = min(np.nanmin(a_samps[:, dim]), np.nanmin(b_samps[:, dim]))
        dim_max = max(np.nanmax(a_samps[:, dim]), np.nanmax(b_samps[:, dim]))
        plot_range.append([dim_min, dim_max])

    # Base Corner Settings
    shared_kwargs = dict(
        labels=labels,
        range=plot_range,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False, # We disable automatic titles to avoid overlaps
        max_n_ticks=3,
        hist_kwargs=dict(density=True)
    )

    # First
    fig = corner.corner(
        a_samps,
        labels=labels,
        range=plot_range,
        color="orange",
        label_kwargs={"fontsize": 14},
        weights=a_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'orange', 'linewidth': 2, 'density': True}
    )

    # Second
    # We turn off 'fill_contours' for the second one so we can see through it
    corner.corner(
        b_samps,
        fig=fig,
        range=plot_range,
        color="dodgerblue",
        label_kwargs={"fontsize": 14}, 
        weights=b_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True, 
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'dodgerblue', 'linewidth': 2, 'density': True}
    )

    # 7. Add Legend and Title
    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i], lw=4)
            for i in range(len(colors))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    
    true_params = mock_data["true_params"][()]

    real_values = "True Properties:\n\n"
    for i, p in enumerate(true_params.values()): # iterating mock_data dictionary
            real_values += f"{labels[i]}: {p}\n"
    
    box_style = dict(
        boxstyle='round,pad=0.5', # Shapes: 'square', 'round', 'larrow', etc.
        facecolor='wheat',         # Inside color of the box
        edgecolor='orange',        # Border color
        alpha=0.5                  # Transparency (0 to 1)
    )
    
    plt.text(-4.8, 5, s=real_values, fontsize=18, bbox=box_style)
    
    ndim = a_samps.shape[1]
    axes = np.array(fig.axes).reshape((ndim, ndim))
    
    for i in range(ndim):
        ax = axes[i, i]
        
        # 2. Extract stats for Bagpipes (Orange)
        a_p = data1['params'][plot_keys[i]]
        a_val, a_plus, a_minus = a_p['q50'], a_p['q84'] - a_p['q50'], a_p['q50'] - a_p['q16']
        
        # 3. Extract stats for Prospector (Black/Blue)
        b_p = data2['params'][plot_keys[i]]
        b_val, b_plus, b_minus = b_p['q50'], b_p['q84'] - b_p['q50'], b_p['q50'] - b_p['q16']
        
        # Special Case: If it's Av (index 2), apply the 1.086 scale to the text labels too
        if i == 2:
            if scale_dust1:
                a_val, a_plus, a_minus = a_val*1.086, a_plus*1.086, a_minus*1.086
            if scale_dust2:
                b_val, b_plus, b_minus = b_val*1.086, b_plus*1.086, b_minus*1.086
        
        # 4. Create the strings
        # Use \text{} or raw strings to handle the LaTeX formatting
        a_str = f"${a_val:.2f}^{{+{a_plus:.2f}}}_{{-{a_minus:.2f}}}$"
        b_str = f"${b_val:.2f}^{{+{b_plus:.2f}}}_{{-{b_minus:.2f}}}$"

        # 5. Set the title
        # We use a newline \n to stack them. Note: 'y' controls the vertical height.
        # 4. Place individual text objects (Manually colored)
        # x=0.5 centers it. y=1.02 and 1.15 stack them above the plot.
        ax.text(0.5, 1.15, a_str, color="orange", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        ax.text(0.5, 1.02, b_str, color="dodgerblue", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        # 6. Coloring the text
        # To get specific colors for specific lines of the title, 
        # we can use ax.annotate or just rely on the labels in the legend.
        # But for absolute clarity, we can color the whole title block:
        #ax.title.set_color('black') # Or 'darkgrey' to be neutral
    
    
    
    fig.suptitle(f"{label1} vs. {label2}\nMock Galaxy {galaxy_id}", fontsize=24, y=1.0)
    
    fig_path = os.path.join(save_dir, f'{galaxy_id}_corner.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
objid = 9999

comp_dir = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/mock_fit"

p_dir = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/prospector/output"
b_dir = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/mock_fit"

p_data = load_prospector_results(objid, p_dir)
b_data = load_bagpipes_results(objid, b_dir)

mock_params = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/mock_fit/{objid}_mock.npz"
mock_data = np.load(mock_params, allow_pickle=True)

plot_corner_mock(galaxy_id=9999, 
                 data1=b_data, label1="Bagpipes",
                 data2=p_data, label2="Prospector", 
                 mock_data=mock_data,
                 save_dir=comp_dir)

Create SFH from Prospector

In [ ]:
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.utils.plotting import posterior_samples
import matplotlib.pyplot as plt

h5_file = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/prospector/output/9999_mock_test_1779297841_mcmc.h5"

# Load PROSPECTOR results
results, obs, model = reader.results_from(h5_file)

imax = np.argmax(results['lnprobability'])

map_parameters = results['chain'][imax, :].copy()

# Build the MAP dictionary
MAP = {}
for a,b in zip(results['theta_labels'], map_parameters):
    MAP[a] = b

zred = MAP['zred']
logmass = MAP['logmass']
agebins = results['model_params'][8]['init']    # 8 for agebins

print(len(agebins))

dt = 10**agebins[:, 1] - 10**agebins[:, 0]

# Collect logsfr_ratios
logsfr_ratios = np.array([MAP[f"logsfr_ratios_{i}"] for i in range(1, len([k for k in MAP if k.startswith("logsfr_ratios_")])+1)])        
# Convert to SFRs
sfh_best = logsfr_ratios_to_sfrs(logmass, logsfr_ratios, agebins)

# Sample from the chains!
n_steps = results['chain'].shape[0]
# Take 500 weighted posterior samples from the chain
sample_indices = np.random.choice(n_steps, size=500, p=results['weights']/np.sum(results['weights']))
# Get the full set of parameters from the chain
samples = results['chain'][sample_indices, :]

sfh_samples = []
for params_i in samples:
    new_map = {}
    for a,b in zip(results['theta_labels'], params_i):
        new_map[a] = b
    # Get logsfr_samples
    logsfr_sample = np.array([new_map[f"logsfr_ratios_{i}"] for i in range(1, len([k for k in new_map if k.startswith("logsfr_ratios_")])+1)])
    sfh_sample = logsfr_ratios_to_sfrs(logmass, logsfr_sample, agebins)
    sfh_samples.append(sfh_sample)

# Takes the per-pixel percentiles such that the final spectra are not actual spectra of Prospectors parameter space
sfh_lower = np.percentile(sfh_samples, 16, axis=0)
sfh_median = np.percentile(sfh_samples, 50, axis=0)
sfh_upper = np.percentile(sfh_samples, 84, axis=0)

# Convert log age bins to linear time (yr)
bin_edges = 10**agebins  # shape (nbins, 2)

bin_edges *= 1e-9

bin_starts = bin_edges[:, 0]

fig, ax = plt.subplots(figsize=(8, 5))

# Plot the Median SFH
ax.step(bin_starts, sfh_median, where='post', color='black', lw=2, label='Median SFH')

# Plot the MAP SFH
ax.step(bin_starts, sfh_best, where='post', color='crimson', lw=2, label='MAP SFH')

# Plot the Uncertainty (16th-84th percentile)
ax.fill_between(bin_starts, sfh_lower, sfh_upper, step="post", color='gray', alpha=0.4, label='68% Confidence')

# Formatting
ax.set_xlabel('Lookback Time (Gyr)', fontsize=14)
ax.set_ylabel('SFR ($M_\odot yr^{-1}$)', fontsize=14)
#ax.invert_xaxis() # Crucial: Lookback time goes from now (left) to past (right)
ax.legend()
#plt.grid(True, which="both", ls="-", alpha=0.2)
plt.tight_layout()
plt.savefig("/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/plots/mock_fit/9999_sfh_prospector.pdf")
plt.show()